In [1]:
import pandas as pd
import sqlite3
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [2]:
load_dotenv()
host = os.getenv("MYSQL_HOST")
port = os.getenv("MYSQL_PORT")
database = os.getenv("MYSQL_DATABASE")
user = os.getenv("MYSQL_USER")
password = os.getenv("MYSQL_PASSWORD")

In [3]:
engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
)

In [4]:
df = pd.read_csv('Data/users.csv')
df.to_sql('users', con=engine, if_exists='replace', index=False)

10

In [5]:
df_from_db = pd.read_sql("select * from users", con=engine)
df_from_db.head()

,id,username,age,email
0,1,john_doe,28,john.doe@example.com
1,2,jane_smith,31,jane.smith@example.com
2,3,michael_brown,24,michael.brown@example.com
3,4,emily_davis,27,emily.davis@example.com
4,5,david_wilson,35,david.wilson@example.com


In [6]:
%reload_ext sql
%config SqlMagic.displaycon = False

connection_string = engine.url.render_as_string(hide_password=False)
%sql $connection_string


In [7]:
%%sql
create table if not exists employee (
    id int primary key,
    name text,
    age int,
    email text
)

0 rows affected.


[]

In [8]:
new_df = pd.DataFrame({
    'id':[1,2],
    'name':['John','Rita'],
    'age':[40,35],
    'email':['john@example.com', 'rita@example.com']
})

new_df.to_sql('employee', con=engine, if_exists='replace', index=False)

2

In [10]:
with engine.begin() as conn:
    conn.execute(text("truncate table users"))

for chunk in pd.read_csv('Data/employee.csv', chunksize=3):
    chunk.to_sql('users', con=engine, if_exists='append', index=False)
    

In [11]:
data_dir = 'Data/'

for file in os.listdir(data_dir):
    if file.endswith('.csv'):
        table_name = os.path.splitext(file)[0]
        print(f"Loading {file} into table {table_name}")
        df = pd.read_csv(os.path.join(data_dir, file))
        df.to_sql(table_name, con=engine, if_exists='replace', index=False)

Loading employee.csv into table employee
Loading users.csv into table users
